# Import

In [ ]:
from IPython.display import clear_output

In [ ]:
AUTOGLUON = False
# AUTOGLUON =  True
# !pip install ray==2.10.0 autogluon.tabular ipywidgets catboost==1.2.5
# clear_output()

In [ ]:
import numpy as np
import pandas as pd
import math
!pip install -q scikit-learn==1.5.2
clear_output()

In [ ]:
import datetime
import sys

import matplotlib
import matplotlib as mpl
import matplotlib.cm as cmap
import matplotlib.colors as mpl_colors
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import seaborn as sns

def hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16)/255 for i in (0, 2, 4))

# palette = ['#b4d2b1', '#568f8b', '#1d4a60', '#cd7e59', '#ddb247', '#d15252']
# palette_rgb = [hex_to_rgb(x) for x in palette]
# cmap = mpl_colors.ListedColormap(palette_rgb)
# colors = cmap.colors
bg_color= '#fdfcf6'

black, red, green, blue = ['#000000', '#ff0000', '#00ff00', '#0000ff']

custom_params = {
    "axes.spines.right": False,
    "axes.spines.top": False,
    'grid.alpha':0.3,
    'figure.figsize': (16, 6),
    'axes.titlesize': 'Large',
    'axes.labelsize': 'Large',
    'figure.facecolor': bg_color,
    'axes.facecolor': bg_color
}

sns.set_theme(
    style='whitegrid',
#     palette=sns.color_palette(palette),
    rc=custom_params
)

from plotly.offline import init_notebook_mode, iplot, plot
import plotly.express as px
import plotly as py
#init_notebook_mode(connected=True)
import plotly.graph_objs as go

import scipy.stats as st

from warnings import simplefilter
simplefilter("ignore")

import random
import os

SEED = 2024
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
seed_everything(SEED)

from IPython.display import clear_output
from tqdm import tqdm, trange

from sklearn.linear_model import *
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from lightgbm import early_stopping, log_evaluation


from lightgbm import LGBMClassifier, LGBMRegressor
from lightgbm import early_stopping, log_evaluation
from sklearn.linear_model import LogisticRegression
import catboost
from xgboost import XGBClassifier

from sklearn.pipeline import make_pipeline, Pipeline

# Encoders
from sklearn.preprocessing import *
from category_encoders.leave_one_out import LeaveOneOutEncoder 
from category_encoders import TargetEncoder, WOEEncoder

# Scalers
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler, Normalizer

from sklearn.ensemble import *
from sklearn.compose import *

from scipy.stats.mstats import gmean, hmean
from scipy.stats import mode
from numpy import mean, median

import re

from sklearn.model_selection import *
from sklearn.metrics import *
from sklearn.base import clone
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV
from sklearn.feature_selection import *
from sklearn.metrics import root_mean_squared_log_error

# if not AUTOGLUON :
#     import eli5
#     from eli5.sklearn import PermutationImportance
#     import shap

from termcolor import colored

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, Sequential
from tensorflow.keras import backend as K

from sklearn.inspection import PartialDependenceDisplay

# &#129300; PROBLEM

In [ ]:
PROBLEM = 'regression'

TARGET = 'Premium Amount'

def load_datasets():
    train = pd.read_csv('/kaggle/input/playground-series-s4e12/train.csv')
    test = pd.read_csv('/kaggle/input/playground-series-s4e12/test.csv')
    sample_sub = pd.read_csv('/kaggle/input/playground-series-s4e12/sample_submission.csv')
    original = pd.read_csv('/kaggle/input/insurance-premium-prediction/Insurance Premium Prediction Dataset.csv')
    
    train = train.drop(['id'], axis=1)
    test = test.drop(['id'], axis=1)

    train['nans'] = train.isnull().sum(axis=1).astype(float)
    test['nans'] = test.isnull().sum(axis=1)
    original['nans'] = original.isnull().sum(axis=1)

    return train, test, original, sample_sub # don't use original

def SCORE(y_true, y_pred):
    return root_mean_squared_log_error(y_true, y_pred)

def LOSS(y_true, y_pred):
    return root_mean_squared_log_error(y_true, np.clip(y_pred, 20, 4999))
    

SCORE_NAME = 'RMSLE'

OBJ = -1 # 1 maximize score, -1 minimize score

# &#128297; ENGINE

## &#128190; Database

In [ ]:
import shutil

INPUT = '/kaggle/input/insurance-4-12/'

class DatasetCollection():
    '''
        Save and load datasets, i.e. (train, test)
    '''
    
    def __init__(self, dir: str = 'datasets') -> None :
        self.dir_ = dir 
        if not os.path.exists(self.dir_) : 
            os.mkdir(self.dir_)

        self.data_ = {}
        
    def __str__(self) -> str :
        s = ''
        for name, ds in self.data_.items() :
            s += f"{name}, {ds['description']} :\n"
            for df in ['train', 'test'] :
                if ds[df] is not None:
                    s += f'{df}, {ds[df].shape[0]} rows {ds[df].shape[1]} columns\n'
            s += '\n'
        return s
    
    def __repr__(self) -> str :
        return 'DatasetCollection :\n\n' + self.__str__()
        
    def load(self) -> None :
        path = os.path.join(INPUT, self.dir_)
        if not os.path.exists(path) :
            print('no datasets found')
            return
        listdir = os.listdir(path)
        for name in listdir :
            self.data_[name] = {
                'train' : pd.read_csv(os.path.join(path, name, 'train.csv')) if os.path.exists(os.path.join(path, name, 'train.csv')) else None,
                'test' : pd.read_csv(os.path.join(path, name, 'test.csv')) if os.path.exists(os.path.join(path, name, 'test.csv')) else None,
            }
            if os.path.exists(os.path.join(path, name, 'description.txt')):
                f = open(os.path.join(path, name, 'description.txt'), 'r')
                self.data_[name]['description'] = f.read()
                f.close()
            else:
                self.data_[name]['description'] = ''
                
    def save(self) -> None :
        for name, ds in self.data_.items() :
            if not os.path.exists(os.path.join(self.dir_, name)) : 
                os.mkdir(os.path.join(self.dir_, name))
            df = self.data_[name]['train']
            if df is not None :
                df.to_csv(os.path.join(self.dir_, name, 'train.csv'), index=False)
            df = self.data_[name]['test']
            if df is not None :
                df.to_csv(os.path.join(self.dir_, name, 'test.csv'), index=False)
            f = open(os.path.join(self.dir_, name, 'description.txt'), 'w')
            f.write(self.data_[name]['description'])
            f.close()
        
    def put(self, name: str = 'loaded', train: pd.DataFrame = None, test: pd.DataFrame = None, description: str = '',) -> None :
        self.data_[name] = {
            'train' : train.copy() if train is not None else None,
            'test'  : test.copy() if test is not None else None,
            'description' : description,
        }
    
    def train(self, name: str = 'loaded') -> pd.DataFrame :
        df = self.data_[name]['train']
        if df is None :
            return None
        return df.copy()
    
    def test(self, name: str = 'loaded') -> pd.DataFrame :
        df = self.data_[name]['test']
        if df is None :
            return None
        return df.copy()
    
    def X_test(self, name: str = 'loaded') -> pd.DataFrame :
        df = self.data_[name]['test']
        if df is None :
            return None
        return df.copy()
    
    def X(self, name: str = 'loaded') -> pd.DataFrame :
        df = self.data_[name]['train'].copy()
        if df is None :
            return None
        cols = df.columns.tolist()
        if TARGET in cols:
            cols.remove(TARGET)
        return df[cols].copy()
    
    def y(self, name: str = 'loaded') -> pd.Series :
        df = self.data_[name]['train'].copy()
        if df is None :
            return None
        cols = df.columns.tolist()
        if TARGET in cols:
            return df[TARGET]
        return None

    def union(self, name: str = 'loaded') -> pd.DataFrame :
        return pd.concat([self.X(name), self.test(name)])
    
    def names(self) -> list :
        return list(self.data_.keys())

    def summary(self, name : str = 'loaded', dataframes : list = ['train', 'test'], tab : bool = True, plots : bool = False, info : bool = True, nrows : int = 3) -> None :
        for df_name in dataframes :
            df = self.data_[name][df_name]
            print(colored(f'\n---------- {name} {df_name} ----------:\n', 'red'))
                    
            if tab:
                display(df.head(nrows))
                display(df.tail(nrows))
                print(colored(f'{name} has {df.shape[0]} rows, {df.shape[1]} columns\n', 'blue'))
        
            inf = pd.DataFrame(df.dtypes).reset_index().rename(columns={'index':'column', 0:'type'})
            df_missed = pd.DataFrame(df.isnull().sum()).reset_index().rename(columns={'index':'column', 0:'missed'})
            df_unique = pd.DataFrame(df.nunique()).reset_index().rename(columns={'index':'column', 0:'unique'})
            inf['missed'] = df_missed['missed']
            inf['unique'] = df_unique['unique']
            inf['duplicate'] = df.duplicated().sum()
            
            desc = pd.DataFrame(df.describe(include='all').transpose())
            if 'min' in desc.columns.tolist():
                inf['min'] = desc['min'].values
                inf['max'] = desc['max'].values
                inf['avg'] = desc['mean'].values
                inf['std dev'] = desc['std'].values
            if 'top' in desc.columns.tolist():
                inf['top value'] = desc['top'].values
                inf['Freq'] = desc['freq'].values    
            
            if info:
                display(inf.style.background_gradient(subset='missed', cmap='Reds').background_gradient(subset='unique', cmap='Greens'))
          
            if plots:
                print()
                if df_missed['missed'].sum() > 0:
                    fig, ax = plt.subplots(1, 1, figsize=(24, 5))
                    sns.barplot(df_missed[df_missed['missed'] > 0], x='column', y='missed', ax=ax)
                    ax.set_title(f'{name} missed values') 
                    ax.bar_label(ax.containers[0])
                    plt.tight_layout()
                    plt.show()
        
                fig, ax = plt.subplots(1, 1, figsize=(24, 5))
                sns.barplot(df_unique[df_unique['unique'] > 0], x='column', y='unique', ax=ax)
                ax.set_title(f'{name} unique values')
                ax.bar_label(ax.containers[0])
                plt.tight_layout()
                plt.show()



import json
names = {}

class CrossValidation() :

    def __init__(
        self,
        name : str               = 'cv',
        estimator_class  : str   = 'unknown',
        dataset_name : str       = 'unknown',
        params : dict            = {},
        n_splits : int           = 5, 
        n_repeats : int          = 1, 
        oof_train                = None, 
        oof_true                 = None, 
        oof_test                 = None, 
        submission               = None,
        oof_scores : list        = [],
        description : str        = '',
        iteration_time : float   = 0.0,
    ) -> None :

        self.summary = {
            'name' : name,
            'estimator_class' : estimator_class,
            'dataset_name' : dataset_name,
            'params' : params,
            'n_splits' : n_splits,
            'n_repeats' : n_repeats,
            'description' : description,
            'scores' : oof_scores,
        }
        self.oof = {
            'train'        : oof_train,
            'test'         : oof_test,
            'true'         : oof_true,
        }

    def __str__(self) -> str :
        s = ''
        for k, v in self.summary.items() :
            if k != 'params':
                s += '    ' + k + ' '* (20-len(k)) + ':' + str(v) + '\n'
        s += 'params:\n'
        p = self.summary['params']
        if type(p) is str :
             s += p + '\n\n'
        else :
            for k, v in p.items() :
                s += '    ' + k + ' '* (20-len(k)) + ':' + str(v) + '\n'
        for df_name, df in self.oof.items() :
            if df is not None:
                s += f'oof {df_name}, {df.shape[0]} rows\n'
        s += '\n'
        return s
    
    def __repr__(self) -> str :
        return 'CrossValidation:\n' + self.__str__()

    def load(self, dir : str) -> bool :
        if not os.path.exists(os.path.join(dir, 'summary.json')):
            print('directory', dir, 'has no summary')
            return False

        f = open(os.path.join(dir, 'summary.json'), 'r')
        summ = f.read()
        f.close()
        self.summary = json.loads(summ)
        
        for df_name, df in self.oof.items() :
            self.oof[df_name] = pd.read_csv(os.path.join(dir, f'{df_name}.csv')).to_numpy().flatten() if os.path.exists(os.path.join(dir, f'{df_name}.csv')) else None
        return True

    def save(self, dir : str) -> None :
        name = self.summary['name']
        if not os.path.exists(os.path.join(dir, name)) : 
            os.mkdir(os.path.join(dir, name))

        for df_name, arr in self.oof.items() :
            if arr is not None:
                df = pd.DataFrame()
                df['value'] = arr
                df.to_csv(os.path.join(dir, name, f'{df_name}.csv'), index=False)

        summary = json.dumps(self.summary, indent=4)
        f = open(os.path.join(dir, name, 'summary.json'), 'w')
        f.write(summary)
        f.close()

    def display(self, estimator, verbose = 2) -> None :
        estimator.upload_cv(self)
        estimator.display_cv_results()
        if verbose > 1 :
            estimator.display_cv_plots()

    def run(self, estimator, db) :
        '''
            rerun cv, put new cv into db
            return new cv
        '''
        s = self.summary
        cv = estimator.crossvalidate(
            db.datasets.train(s['dataset_name']),
            db.datasets.test(s['dataset_name']),
            name = s['name'],
            description = s['description'],
            n_splits = s['n_splits'],
            n_repeats = s['n_repeats'],
            dataset_name = s['dataset_name'],
        )
        db.cvs.put(cv)
        return cv

    def submit(self) :
        '''
            oof test -> .csv
        '''
        sub = sample_sub.copy()
        sub[TARGET] = self.oof['test_pred']
        sub.to_csv(f"cv_{self.summary['name']}.csv", index=False)
        display(sub.head(3))



class CVCollection():
    '''
        Save and load CV settings, i.e. model, perameters, number of
        splits and repeats, and OOF data.
    '''
    
    def __init__(self, dir: str = 'cvs') -> None :
        self.dir_ = dir 
        if not os.path.exists(self.dir_) : 
            os.mkdir(self.dir_)

        self.data_ = {}
        
    def __str__(self) -> str :
        s = ''
        for _, cv in self.data_.items() :
            s += str(cv) + '\n'
        return s
    
    def __repr__(self) -> str :
        return 'CVCollection :\n\n' + self.__str__()

    def load(self) -> None :
        '''
            load cvs from input directory
        '''
        path = os.path.join(INPUT, self.dir_)
        if not os.path.exists(path) :
            print('no datasets found')
            return
        listdir = os.listdir(path)
        for name in listdir :
            cv = CrossValidation()
            ok = cv.load(os.path.join(path, name))
            if ok :
                self.data_[name] = cv
                
    def save(self) -> None :
        '''
             save all cvs
        '''
        for name, cv in self.data_.items() :
            cv.save(os.path.join(self.dir_))
        
    def put(self, cv : CrossValidation) -> None :
        '''
            put cv into database
        '''
        self.data_[cv.summary['name']] = cv

    def get(self, cv_name : str) -> CrossValidation :
        '''
            returns cv by name
        '''
        if not cv_name in self.data_:
            print(colored(f'cv {cv_name} not found'), 'red')
            return None
        return self.data_[cv_name]

class Database():
    '''
        Save database when notebook saved with commit,
        load on run.
    '''
    
    def __init__(self, dir: str = 'database') -> None :
        self.dir_ = dir 
        if not os.path.exists(self.dir_) : 
            os.mkdir(self.dir_)
       
        self.datasets = DatasetCollection(dir = os.path.join(self.dir_, 'datasets'))
        self.cvs = CVCollection(dir = os.path.join(self.dir_, 'cvs'))
        
    def __str__(self) -> str :
        s = '\nDatasets:\n' + self.datasets.__str__()
        s += '\nCVs:\n' + self.cvs.__str__()
        return s
    
    def __repr__(self) -> str :
        return 'Database :\n\n' + self.__str__()
        
    def load(self) -> None :
        self.datasets.load()
        self.cvs.load()
        
    def save(self) -> None :
        self.datasets.save()
        self.cvs.save()        
        shutil.make_archive(self.dir_, 'zip', os.path.join('/kaggle/working', self.dir_))

## ⚖ Averager
We will use it to average different prediction in fold with weights optimized 

In [ ]:
from functools import partial
import scipy as sp

class Averager(object):

    def __init__(self, method='nelder-mead', round_avg=False, options={}):
        self.weights_ = []
        self.opt_ = ''
        self.method_ = method
        self.round_avg_ = round_avg
        self.options_ = options

    def _weighted_average(self, weights, values):
        qty = len(values)
        sum_values = values[0] * weights[0]
        sum_weights = weights[0]
        for i in range(1, qty):
            sum_values += values[i] * weights[i]
            sum_weights += weights[i]
        if self.round_avg_:
            return int(np.round(sum_values / sum_weights, 0))
        return sum_values / sum_weights

    def _score(self, weights, values, true_labels):
        preds = self._weighted_average(weights, values)
        return LOSS(true_labels, preds)

    def fit(self, values, true_labels):
        qty = len(values)
        initial_weights = [1 for _ in range(qty)]
        score_partial = partial(self._score, values=values, true_labels=true_labels)
        self.opt_ = sp.optimize.minimize(score_partial, initial_weights, method=self.method_, options=self.options_)
        self.weights_ = self.opt_['x']

    def predict(self, values):
        assert len(self.weights_) == len(values), 'Averager error, must be fitted before predict'
        return self._weighted_average(self.weights_, values)

    def fit_predict(self, values, true_labels):
        self.fit(values, true_labels)
        return self.predict(values)

    def weights(self):
        return self.weights_

    def optimization(self):
        return self.opt_



## &#128188; Estimators Classes

In [ ]:
class Estimator():

    def __init__(self, name : str = 'model', params : list = {}, verbose : int = 1) :
        self.name = name
        self.models_ = []
        self.model_ = None
        self.params_ = params
        self.verbose_ = verbose
        
        self.cv = None

    def fit(self, X, y):
        pass

    def fit_predict(self, X, y, X_val, y_val):
        pass

    def fit_predict_proba(self, X, y, X_val, y_val):
        pass

    def predict_proba(self, X):
        pass

    def predict(self, X):
        pass

    def crossvalidate(
            train_: pd.DataFrame, 
            test_:  pd.DataFrame,
            n_splits=5, n_repeats=1, random_state=42, verbose=1, dataset_name='unknown', use_tqdm=True, clear=True,
        ) -> CrossValidation :
        pass
    
    def display_cv_plots():
        pass
    
    def display_cv_results():
        pass
    
    def upload_cv(self, cv : CrossValidation) -> None :
        self.cv = cv
    
    def submit():
        pass
  

In [ ]:
class Regressor(Estimator):

    def fit_predict(self, X, y, X_val, y_val):
        self.fit(X, y, X_val, y_val)
        return self.predict(X_val)

    def fit_predict_proba(self, X, y, X_val, y_val):
        self.fit(X, y, X_val, y_val)
        return self.predict(X_val)

    def predict_proba(self, X):
        return self.predict(X)
    
    def predict(self, X):
        assert self.model_ is not None, 'Model error, must be fitted before predict'
        return self.model_.predict(X)
        
    def crossvalidate(self,
            train_: pd.DataFrame, 
            test_:  pd.DataFrame,
            name : str = None,            
            description : str = '',
            n_splits=5, n_repeats=1, random_state=42, verbose=2, dataset_name='unknown', use_tqdm=True, clear=True,
        ) -> CrossValidation :

        if name is None :
            name = self.name
        
        # debug
        train, test = train_.copy(), test_.copy()
        train_n_rows = train.shape[0]
        if verbose > 0 :
            print(f'\n---------- cv: train: {train.shape}, test: {test.shape} -----------\n')

        features = test.columns.to_list()

        folds = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)

        oof_test = np.zeros(len(test))
        oof_train = np.zeros(len(train))
        oof_true = np.zeros(len(train))
        oof_scores = []

        assert train_n_rows == len(oof_train) and train_n_rows == len(oof_true), f"train ({train_n_lows}), oof_test_proba ({len(oof_train_proba)}), oof_true ({len(oof_true)}) doesn't match"
        
        if use_tqdm and verbose > 0:
            data = tqdm(enumerate(folds.split(train[features], train[TARGET])), desc='Fold', total=n_splits*n_repeats, file=sys.stdout, colour='GREEN')
        else:
            data = enumerate(folds.split(train[features], train[TARGET]))

        start = datetime.datetime.now()

        for i, (train_idx,val_idx) in data:

            train_labels =  train.loc[train_idx, TARGET]
            val_labels =  train.loc[val_idx, TARGET]
            train_features = train.loc[train_idx, features]
            val_features = train.loc[val_idx, features]

            train_labels_log = np.log1p(train_labels)
            val_labels_log = np.log1p(val_labels)
    
            val_preds_log =  self.fit_predict(train_features, train_labels_log, val_features, val_labels_log)
            test_preds_log = self.predict(test)

            val_preds = np.clip(np.expm1(val_preds_log), a_min = 20.0, a_max = 4999.0)
            test_preds = np.clip(np.expm1(test_preds_log), a_min = 20.0, a_max = 4999.0)

            oof_test += test_preds

            score = SCORE(val_labels, val_preds)
            oof_scores.append(score)
            
            oof_train[val_idx] = val_preds
            oof_true[val_idx] = val_labels
            assert train_n_rows == len(oof_train) and train_n_rows == len(oof_true), f"train ({train_n_lows}), oof_test_proba ({len(oof_train_proba)}), oof_true ({len(oof_true)}) doesn't match"

            if clear:
                clear_output(wait=True)
 
            if verbose > 0:
                print('\nfold', i, SCORE_NAME, score, '\n')

        if clear:
            clear_output(wait=True)

        iteration_time = (datetime.datetime.now() - start).total_seconds() / n_splits * n_repeats

        oof_test /= n_splits * n_repeats

        assert train_n_rows == len(oof_train) and train_n_rows == len(oof_true), f"train ({train_n_lows}), oof_test_proba ({len(oof_train_proba)}), oof_true ({len(oof_true)}) doesn't match"
        
        self.cv = CrossValidation(
            name                = name,
            estimator_class     = self.name,
            dataset_name        = dataset_name,
            params              = self.params_,
            n_splits            = n_splits, 
            n_repeats           = n_repeats, 
            oof_train           = oof_train, 
            oof_true            = oof_true, 
            oof_test            = oof_test, 
            oof_scores          = oof_scores,
            description         = description,
            iteration_time      = iteration_time,            
        )

        if verbose > 0:
            self.display_cv_results()
        if verbose > 1:
            self.display_cv_plots()
        return self.cv
    
    def cv_scores(self):       
        oof_train = self.cv.oof['train']
        oof_true = self.cv.oof['true']

        if 'scores' in self.cv.summary :
            mean_oof_score = np.mean(self.cv.summary['scores'])
        else:
            mean_oof_score = None
            
        score = SCORE(oof_true, oof_train)
        R2 = r2_score(oof_true, oof_train)
        
        self.cv_scores_ = {
            'Model': self.name,
            'Dataset': self.cv.summary['dataset_name'] if 'dataset_name' in self.cv.summary else 'n/a',
            f'Mean OOF {SCORE_NAME}': mean_oof_score,
            f'{SCORE_NAME}': score,
            'R2': R2,
            'iteration_time': self.cv.summary['iteration_time'] if 'iteration_time' in self.cv.summary else 'n/a',
        }    
        return self.cv_scores_        

    def display_cv_results(self):
        scores = self.cv_scores()
        print(colored(f'\n---------- {self.name} {SCORE_NAME}: {scores[SCORE_NAME]} ----------:\n', 'red'))
        display(pd.DataFrame([scores,]))
                
    def display_cv_plots(self):
        
        scores = self.cv_scores()
        
        fig, axs = plt.subplots(1, 3, figsize=(20, 10))
        axs = axs.flatten()

        if 'scores' in self.cv.summary:
            sns.boxplot(self.cv.summary['scores'], ax=axs[0])
            axs[0].set_title(f'OOF {SCORE_NAME}')
        else:
            axs[0].set_title(f'OOF {SCORE_NAME} N/A')

        df = pd.DataFrame()
        df['Actual'] = self.cv.oof['true']
        df['Predicted'] = self.cv.oof['train']
        sns.scatterplot(df, x='Actual', y='Predicted', ax=axs[1])
        sns.lineplot(x=[0, np.max(self.cv.oof['train'])], y=[0, np.max(self.cv.oof['train'])], ax=axs[1], color=red)
        axs[1].set_title('Actual vs Predicted')

        d = PredictionErrorDisplay.from_predictions(np.array(self.cv.oof['true']), np.array(self.cv.oof['train']), ax=axs[2])
        axs[2].set_title('Prediction error')
   
        plt.tight_layout()
        plt.show()     

    def submit(self):
        sub = sample_sub.copy()
        sub[TARGET] = self.cv.oof['test']
        score = self.cv_scores()[SCORE_NAME]
        sub.to_csv(f"{self.name}_{self.cv.summary['name']}_{score:.5f}.csv", index=False)
        display(sub.head(30))

In [ ]:

class Classifier(Estimator):

    def fit_predict_proba(self, X, y, X_val, y_val):
        self.fit(X, y, X_val, y_val)
        return self.predict_proba(X_val)
        
    def predict(self, X):
        assert self.model_ is not None, 'Model error, must be fitted before predict'
        return np.rint(self.predict_proba(X)).astype(int)

    def predict_proba(self, X):
        assert self.model_ is not None, 'Model error, must be fitted before predict'
        return self.model_.predict_proba(X)[:, -1]
        
    def crossvalidate(self,
            train_: pd.DataFrame, 
            test_:  pd.DataFrame,
            name : str = None,            
            description : str = '',
            n_splits=5, n_repeats=1, random_state=42, verbose=2, dataset_name='unknown', use_tqdm=True, clear=True,
        ) -> CrossValidation :

        if name is None :
            name = self.name
        
        # debug
        train, test = train_.copy(), test_.copy()
        train_n_rows = train.shape[0]
        if verbose > 0 :
            print(f'\n---------- cv: train: {train.shape}, test: {test.shape} -----------\n')

        features = test.columns.to_list()

        folds = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)

        test_proba_mean = np.zeros(len(test))
        oof_train_proba = np.zeros(len(train))
        oof_true        = np.zeros(len(train))
                 
        oof_scores = []

        assert train_n_rows == len(oof_train_proba) and train_n_rows == len(oof_true), f"train ({train_n_lows}), oof_test_proba ({len(oof_train_proba)}), oof_true ({len(oof_true)}) doesn't match"
        
        if use_tqdm and verbose > 0:
            data = tqdm(enumerate(folds.split(train[features], train[TARGET])), desc='Fold', total=n_splits*n_repeats, file=sys.stdout, colour='GREEN')
        else:
            data = enumerate(folds.split(train[features], train[TARGET]))

        start = datetime.datetime.now()

        for i, (train_idx,val_idx) in data:

            train_labels =  train.loc[train_idx, TARGET]
            val_labels =  train.loc[val_idx, TARGET]
            train_features = train.loc[train_idx, features]
            val_features = train.loc[val_idx, features]
    
            val_proba =  self.fit_predict_proba(train_features, train_labels, val_features, val_labels)
            test_proba = self.predict_proba(test)

            test_proba_mean += test_proba    

            score = SCORE(val_labels, val_proba)
            oof_scores.append(score)
            
            oof_train_proba[val_idx] = val_proba
            oof_true[val_idx] = val_labels
            assert train_n_rows == len(oof_train_proba) and train_n_rows == len(oof_true), f"train ({train_n_lows}), oof_test_proba ({len(oof_train_proba)}), oof_true ({len(oof_true)}) doesn't match"

            if clear:
                clear_output(wait=True)
 
            if verbose > 0:
                print('\nfold', i, SCORE_NAME, score, '\n')

        if clear:
            clear_output(wait=True)

        iteration_time = (datetime.datetime.now() - start).total_seconds() / n_splits * n_repeats

        test_proba_mean /= n_splits * n_repeats

        assert train_n_rows == len(oof_train_proba) and train_n_rows == len(oof_true), f"train ({train_n_lows}), oof_test_proba ({len(oof_train_proba)}), oof_true ({len(oof_true)}) doesn't match"
        
        self.cv = CrossValidation(
            name                = name,
            estimator_class     = self.name,
            dataset_name        = dataset_name,
            params              = self.params_,
            n_splits            = n_splits, 
            n_repeats           = n_repeats, 
            
            oof_train_proba     = oof_train_proba, 
            oof_true            = oof_true, 
            oof_test_proba      = test_proba_mean, 
            
            oof_scores          = oof_scores,
            description         = description,
            iteration_time      = iteration_time,            
        )

        if verbose > 0:
            self.display_cv_results()
        if verbose > 1:
            self.display_cv_plots()
        return self.cv

    def decision(self, X) -> np.ndarray :
        return np.rint(X).astype(int)
    
    def cv_scores(self):       
        oof_train_proba = self.cv.oof['train_proba']
        oof_true = self.cv.oof['true']

        if 'scores' in self.cv.summary :
            mean_oof_score = np.mean(self.cv.summary['scores'])
        else:
            mean_oof_score = None
            
        prediction = self.decision(oof_train_proba)
        score = SCORE(oof_true, oof_train_proba)

        model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(oof_true, prediction, average="weighted")
        model_precision, model_recall, model_f1 = round(model_precision, 4), round(model_recall, 4), round(model_f1, 4)
        model_matthews_corrcoef = round(matthews_corrcoef(oof_true, prediction), 4)

        self.cv_scores_ = {
            'Model': self.name,
            'Dataset': self.cv.summary['dataset_name'] if 'dataset_name' in self.cv.summary else 'n/a',
            f'Mean OOF {SCORE_NAME}': mean_oof_score,
            f'{SCORE_NAME}': score,
            'Accuracy': accuracy_score(oof_true, prediction),
            'Precision Score': model_precision,
            'Recall Score': model_recall,
            'f1 Score': model_f1,
            'Matthews Corr Coef': model_matthews_corrcoef,
            'iteration_time': self.cv.summary['iteration_time'] if 'iteration_time' in self.cv.summary else 'n/a',
        }    
        return self.cv_scores_        

    def display_cv_results(self):
        scores = self.cv_scores()
        print(colored(f'\n---------- {self.name} {SCORE_NAME}: {scores[SCORE_NAME]} ----------:\n', 'red'))
        display(pd.DataFrame([scores,]))
                
    def display_cv_plots(self):
        scores = self.cv_scores()
        fig, axs = plt.subplots(2, 3, figsize=(20, 10))
        axs = axs.flatten()

        if 'scores' in self.cv.summary:
            sns.boxplot(self.cv.summary['scores'], ax=axs[0])
            axs[0].set_title(f'OOF {SCORE_NAME}')
        else:
            axs[0].set_title(f'OOF {SCORE_NAME} N/A')

        prediction = self.decision(self.cv.oof['train_proba'])
        confusion = confusion_matrix(self.cv.oof['true'], prediction)
        labels = ['class 0', 'Class 1']
        sns.heatmap(confusion, annot=True, annot_kws={"fontsize":24}, fmt=",d", xticklabels=labels, yticklabels=labels, cmap='plasma', cbar=False, ax=axs[1])
        axs[1].set_title(f'Prediction')
        axs[1].set_ylabel("Actual Class")
        axs[1].set_xlabel("Predicted Class")    

        RocCurveDisplay.from_predictions(self.cv.oof['true'], self.cv.oof['train_proba'], ax=axs[2])
        axs[2].set_title('ROC')
        CalibrationDisplay.from_predictions(self.cv.oof['true'], np.array(self.cv.oof['train_proba']).clip(0, 1), n_bins=30, strategy='quantile', ax=axs[3])
        axs[3].set_title('Calibration')

        PrecisionRecallDisplay.from_predictions(self.cv.oof['true'], np.array(self.cv.oof['train_proba']).clip(0, 1), ax=axs[4])
        axs[4].set_title('Precision-Recall')
        
        fpr, fnr, thresholds = det_curve(self.cv.oof['true'], self.cv.oof['train_proba'])
        ax = axs[5]
        sns.lineplot(x=thresholds, y=fpr, label=f'FPR', ax=ax, color='navy')
        sns.lineplot(x=thresholds, y=fnr, label=f'FNR', ax=ax, color='red')

        # fr = fpr + fnr
        # thr = thresholds[fr.argmin()]       
        
        # sns.lineplot(x=thresholds, y=fnr+fpr, label=f'FR', ax=ax, color='green')
        # sns.lineplot(x=[thr, thr], y=[0, fr.min()], ax=ax, color='green')
        
        ax.set_title('FPR-FNR curves')
        ax.set_xlabel('Threshold')
        ax.set_ylabel('Error Rate')

        plt.tight_layout()
        plt.show()     

    def submit(self):
        sub = sample_sub.copy()
        sub[TARGET] = self.decision(self.cv.oof['test_proba'])
        score = self.cv_scores()[SCORE_NAME]
        sub.to_csv(f"{self.name}_{score}.csv", index=False)
        display(sub.head(30))
        

In [ ]:
        
class Multiclass(Classifier) :

    def fit_predict_proba(self, X, y, X_val, y_val):
        self.fit(X, y, X_val, y_val)
        return self.predict_proba(X_val)
        
    def predict(self, X):
        return np.rint(self.predict_proba(X)).astype(int)

    def predict_proba(self, X):
        assert self.model_ is not None, 'Model error, must be fitted before predict'
        return self.model_.predict_proba(X)
        
    def crossvalidate(self,
            train_: pd.DataFrame, 
            test_:  pd.DataFrame,
            name : str = None,            
            description : str = '',
            n_splits=5, n_repeats=1, random_state=42, verbose=2, dataset_name='unknown', use_tqdm=True, clear=True,
        ) -> CrossValidation :

        if name is None :
            name = self.name
        
        # debug
        train, test = train_.copy(), test_.copy()
        train_n_rows = train.shape[0]
        if verbose > 0 :
            print(f'\n---------- cv: train: {train.shape}, test: {test.shape} -----------\n')

        self.classes = train[TARGET].unique()
        self.labels = [f'class {i}' for i in self.classes]

        self.target_encoder = LabelEncoder()
        train[TARGET] = self.target_encoder.fit_transform(train[TARGET])

        features = test.columns.to_list()

        folds = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)

        oof_test = []
        oof_train = []
        oof_true = []
        oof_scores = []
        
        if use_tqdm and verbose > 0:
            data = tqdm(enumerate(folds.split(train[features], train[TARGET])), desc='Fold', total=n_splits*n_repeats, file=sys.stdout, colour='GREEN')
        else:
            data = enumerate(folds.split(train[features], train[TARGET]))

        start = datetime.datetime.now()

        for i, (train_idx,val_idx) in data:

            train_labels =  train.loc[train_idx, TARGET]
            val_labels =  train.loc[val_idx, TARGET]
            train_features = train.loc[train_idx, features]
            val_features = train.loc[val_idx, features]
    
            val_proba =  self.fit_predict_proba(train_features, train_labels, val_features, val_labels)
            test_proba = self.predict_proba(test)

            oof_test.append(test_proba)

            val_prediction = np.rint(val_proba).astype(int)
            score = SCORE(val_labels, self.prediction(val_proba))
            oof_scores.append(score)

            oof_train.extend(val_proba)
            oof_true.extend(val_labels)

            if clear:
                clear_output(wait=True)
 
            if verbose > 0:
                print('\nfold', i, SCORE_NAME, score, '\n')

        if clear:
            clear_output(wait=True)

        iteration_time = (datetime.datetime.now() - start).total_seconds() / n_splits * n_repeats

        oof_test = np.mean(oof_test, axis=0)
        
        self.cv = CrossValidation(
            name                = name,
            estimator_class     = self.name,
            dataset_name        = dataset_name,
            params              = self.params_,
            n_splits            = n_splits, 
            n_repeats           = n_repeats, 
            oof_train           = oof_train, 
            oof_true            = oof_true, 
            oof_test            = oof_test, 
            oof_scores          = oof_scores,
            description         = description,
            iteration_time      = iteration_time,            
        )

        if verbose > 0:
            self.display_cv_results()
        if verbose > 1:
            self.display_cv_plots()
        return self.cv

    def prediction(self, X : np.ndarray) -> np.ndarray :
        return np.argmax(X, axis=1)
    
    def cv_scores(self):       
        oof_train_proba = self.cv.oof['train']
        oof_true = self.cv.oof['true']

        if 'scores' in self.cv.summary :
            mean_oof_score = np.mean(self.cv.summary['scores'])
        else:
            mean_oof_score = None
            
        prediction = self.prediction(oof_train_proba).astype(int)
        score = SCORE(oof_true, self.prediction(oof_train_proba))

#         model_accuracy = round(accuracy_score(oof_true, prediction), 4)
        model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(oof_true, prediction, average="weighted")
        model_precision, model_recall, model_f1 = round(model_precision, 4), round(model_recall, 4), round(model_f1, 4)
        model_matthews_corrcoef = round(matthews_corrcoef(oof_true, prediction), 4)

        self.cv_scores_ = {
            'Model': self.name,
            'Dataset': self.cv.summary['dataset_name'] if 'dataset_name' in self.cv.summary else 'n/a',
            f'Mean OOF {SCORE_NAME}': mean_oof_score,
            f'{SCORE_NAME}': score,
#             'Accuracy Score': model_accuracy,
            'Precision Score': model_precision,
            'Recall Score': model_recall,
            'f1 Score': model_f1,
            'Matthews Corr Coef': model_matthews_corrcoef,
            'iteration_time': self.cv.summary['iteration_time'] if 'iteration_time' in self.cv.summary else 'n/a',
        }    
        return self.cv_scores_             
               
    def display_cv_plots(self):
        scores = self.cv_scores()
        fig, axs = plt.subplots(1, 2, figsize=(20, 10))
        axs = axs.flatten()

        if 'scores' in self.cv.summary:
            sns.boxplot(self.cv.summary['scores'], ax=axs[0])
            axs[0].set_title(f'OOF {SCORE_NAME}')
        else:
            axs[0].set_title(f'OOF {SCORE_NAME} N/A')

        prediction = self.prediction(self.cv.oof['train'])
        
        confusion = confusion_matrix(self.cv.oof['true'], prediction)
        
        sns.heatmap(confusion, annot=True, annot_kws={"fontsize":24}, fmt=",d", xticklabels=self.labels, yticklabels=self.labels, cmap='plasma', cbar=False, ax=axs[1])
        axs[1].set_title(f'Prediction')
        axs[1].set_ylabel("Actual Class")
        axs[1].set_xlabel("Predicted Class")    

        plt.tight_layout()
        plt.show()     

    def submit(self):
        sub = sample_sub.copy()
        sub[TARGET] = self.target_encoder.inverse_transform(self.prediction(self.cv.oof['test']))
        score = self.cv_scores()[SCORE_NAME]
        sub.to_csv(f"{self.name}_{score:.5f}.csv", index=False)
        display(sub.head(3))
   

In [ ]:
 
class RegressorWrapper(Regressor):
    
    def __init__(self, model, name='model', params={}, verbose=1):
        super().__init__(name=name, params=params, verbose=verbose)
        self.model_ = model
        
    def fit(self, X, y, X_val, y_val):
        self.model_.fit(X, y)


class RegressorToClassifierWrapper(Classifier):
    
    def __init__(self, model, name='model', params={}, verbose=1) :
        super().__init__(name=name, params=params, verbose=verbose)
        self.model_ = model
        
    def fit(self, X, y, X_val, y_val):
        self.model_.fit(X, y)
        
    def predict_proba(self, X):
        return self.model_.predict(X)
         
    def predict(self, X):
        assert self.model_ is not None, 'Model error, must be fitted before predict'
        return DECISION(self.model_.predict(X))
   
    
class ClassifierWrapper(Classifier):
    
    def __init__(self, model, name='model', params={}, verbose=1):
        super().__init__(name=name, params=params, verbose=verbose)
        self.model_ = model
        
    def fit(self, X, y, X_val, y_val):
        self.model_.fit(X, y)    
    
class MulticlassWrapper(Multiclass):
    
    def __init__(self, model, name='model', params={}, verbose=1):
        super().__init__(name=name, params=params, verbose=verbose)
        self.model_ = model
        
    def fit(self, X, y, X_val, y_val):
        self.model_.fit(X, y)

In [ ]:

class EnsembleRegressor(Regressor):
    
    def __init__(self, estimators, name='ENS', params={}, verbose=1, options = {}):
        super().__init__(name=name, params=params, verbose=verbose)
        
        self.estimators_ = estimators
        self.options_ = options
        
    def fit_predict(self, X, y, X_val, y_val):
        
        print('\n')

        self.models_ = []

        val_preds = []

        for estimator in self.estimators_:
            try:
                m = clone(estimator)
            except:
                m = estimator
            val_p = m.fit_predict(X, y, X_val, y_val)
            if self.verbose_ > 0:
                print(colored(f'\n{m.__class__.__name__}: {SCORE(y_val, val_p)}', 'blue'))
            self.models_.append(m)
            val_preds.append(val_p)
        
        if self.verbose_ > 0:
            print('\nPREDS:')
            for pred in val_preds:
                print(pred)

            preds_mean = np.mean(val_preds, axis=0)
            print('MEAN:\n', preds_mean)

            print(colored(f'{SCORE_NAME} OF MEAN: {SCORE(y_val, preds_mean)}', 'red'))

        self.averager_ = Averager()
        VAL_PREDS = self.averager_.fit_predict(val_preds, y_val)
        if self.verbose_ > 0:
            print('\nWEIGHTS:\n', self.averager_.weights())
            print('AVGW (weighted average):\n', VAL_PREDS)

            print(colored(f'{SCORE_NAME} OF AVGW: {SCORE(y_val, VAL_PREDS)}\n\n', 'red'))
        
        if 'optimize' in self.options_:
            if not self.options_['optimize']:
                return preds_mean
        return VAL_PREDS
            
    def predict(self, X):
        assert len(self.models_) > 0, 'Model error, must be fitted before predict'
        if len(self.models_) == 1:
            return self.models_[0].predict(X)

        preds = [model.predict(X) for model in self.models_]
        
        if 'optimize' in self.options_:
            if not self.options_['optimize']:
                return np.mean(preds, axis=0)
        return self.averager_.predict(preds)       
    
# EXAMPLE
estimator = EnsembleRegressor(
    [
        RegressorWrapper(LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)) ,
        RegressorWrapper(XGBRegressor(n_estimators=100, random_state=42, enable_categorical=True)) ,
    ], 
    name='ENS',
)

In [ ]:

class EnsembleClassifier(Classifier):
    
    def __init__(self, classifiers, name='ENS', params={}, verbose=1, options = {}):
        super().__init__(name=name, params=params, verbose=verbose)
        
        self.classifiers_ = classifiers
        self.options_ = options
        
    def fit_predict_proba(self, X, y, X_val, y_val):
        
        print('\n')

        self.models_ = []

        val_probas = []

        for classifier in self.classifiers_:
            try:
                m = clone(classifier)
            except:
                m = classifier
            val_p = m.fit_predict_proba(X, y, X_val, y_val)
            if self.verbose_ > 0:
                print(colored(f'\n{m.__class__.__name__}: {-LOSS(y_val, val_p)}', 'blue'))
            self.models_.append(m)
            val_probas.append(val_p)
        
        if self.verbose_ > 0:
            print('\nPROBAS:')
            for proba in val_probas:
                print(proba)

            probas_mean = np.mean(val_probas, axis=0)
            print('MEAN:\n', probas_mean)

            print(colored(f'-LOSS OF MEAN: {-LOSS(y_val, probas_mean)}', 'red'))

        self.averager_ = Averager()
        VAL_PROBAS = self.averager_.fit_predict(val_probas, y_val)
        if self.verbose_ > 0:
            print('\nWEIGHTS:\n', self.averager_.weights())
            print('AVGW (weighted average):\n', VAL_PROBAS)

            print(colored(f'-LOSS OF AVGW: {-LOSS(y_val, VAL_PROBAS)}\n\n', 'red'))
        
        if 'optimize' in self.options_:
            if not self.options_['optimize']:
                return probas_mean
        return VAL_PROBAS

    def predict(self, X) :
        return self.decision(self.predict_proba(X))        
            
    def predict_proba(self, X):
        assert len(self.models_) > 0, 'Model error, must be fitted before predict'
        if len(self.models_) == 1:
            return self.models_[0].predict_proba(X)

        probas = [model.predict_proba(X) for model in self.models_]
        
        if 'optimize' in self.options_:
            if not self.options_['optimize']:
                return np.mean(probas, axis=0)
        return self.averager_.predict(probas)       
    
# EXAMPLE
estimator = EnsembleClassifier(
    [
        ClassifierWrapper(LGBMClassifier(n_estimators=100, random_state=42, verbose=-1, objective='binary', metric='auc')) ,
        ClassifierWrapper(XGBClassifier(n_estimators=100, random_state=42, enable_categorical=True, objective='binary:logistic')) ,
    ], 
    name='ENS',
)

In [ ]:
class StackRegressor(Regressor):
    
    def __init__(self, 
            estimators : list,
            datasets_names : list,
            final_estimator: Estimator,
            name : str = 'STK', 
            params : dict = {}, 
            verbose : int = 2,
        ):
        
        super().__init__(name = name, params = params, verbose = verbose)
        
        self.estimators_ = estimators 
        self.datasets_names_ = datasets_names
        self.final_estimator_ = final_estimator

        count = len(self.estimators_)
        if type(self.datasets_names_) is str :
            self.datasets_names_ = [self.datasets_names_] * count
        
    def crossvalidate(self,
            name : str = None,            
            description : str = '',
            n_splits=5, n_repeats=1, random_state=42, verbose=2, use_tqdm=True, clear=True,
        ) -> CrossValidation :

        if name is None :
            name = self.name
        y = DB.datasets.train(self.datasets_names_[0])[TARGET]
        trn = pd.DataFrame()
        tst = pd.DataFrame()
        trn[TARGET] = y
        for i, estimator in enumerate(self.estimators_) :
            yi = DB.datasets.train(self.datasets_names_[i])[TARGET]
            assert np.array_equal(y, yi, equal_nan=True), 'datasets are not equal'
            cv = estimator.crossvalidate(
                    DB.datasets.train(self.datasets_names_[i]),
                    DB.datasets.test(self.datasets_names_[i]),
                    dataset_name = self.datasets_names_[i], 
                    n_splits = n_splits,
                    n_repeats = n_repeats,
                    clear = False,
                    verbose = 1,
                    name = '',
            )
            trn[f'est{i}'] = cv.oof['train']
            tst[f'est{i}'] = cv.oof['test']

        display(trn)

        display(tst)
            
        self.cv = self.final_estimator_.crossvalidate(
                trn, tst,
                dataset_name = str(self.datasets_names_), 
                n_splits = 10,
                n_repeats = 1,
                clear = clear,
                use_tqdm = use_tqdm,
                verbose = verbose,
                name = name,
        )
        return   self.cv

# EXAMPLE
estimator = StackRegressor(
    estimators = [
        RegressorWrapper(LGBMRegressor(n_estimators=100, random_state=42, verbose=-1), name = 'LGBM'),
        RegressorWrapper(XGBRegressor(n_estimators=100, random_state=42, enable_categorical=True), name = 'XGB'),
    ],
    datasets_names = 'lxe',
    final_estimator = RegressorWrapper(LinearRegression(), name='LR'),
    name = 'STK_LR'
)

In [ ]:

class StackClassifier(Classifier):
    
    def __init__(self, 
            estimators : list,
            datasets_names : list,
            final_estimator: Estimator,
            name : str = 'STK', 
            params : dict = {}, 
            verbose : int = 2,
        ):
        
        super().__init__(name = name, params = params, verbose = verbose)
        
        self.estimators_ = estimators 
        self.datasets_names_ = datasets_names
        self.final_estimator_ = final_estimator

        count = len(self.estimators_)
        if type(self.datasets_names_) is str :
            self.datasets_names_ = [self.datasets_names_] * count
        
    def crossvalidate(self,
            name : str = None,            
            description : str = '',
            n_splits=5, n_repeats=1, random_state=42, verbose=2, use_tqdm=True, clear=True,
        ) -> CrossValidation :

        if name is None :
            name = self.name
        y = DB.datasets.train(self.datasets_names_[0])[TARGET]
        trn = pd.DataFrame()
        tst = pd.DataFrame()
        trn[TARGET] = y
        for i, estimator in enumerate(self.estimators_) :
            yi = DB.datasets.train(self.datasets_names_[i])[TARGET]
            assert np.array_equal(y, yi, equal_nan=True), 'datasets are not equal'
            cv = estimator.crossvalidate(
                    DB.datasets.train(self.datasets_names_[i]),
                    DB.datasets.test(self.datasets_names_[i]),
                    dataset_name = self.datasets_names_[i], 
                    n_splits = n_splits,
                    n_repeats = n_repeats,
                    clear = False,
                    verbose = 1,
                    name = '',
            )
            trn[f'est{i}'] = cv.oof['train']
            tst[f'est{i}'] = cv.oof['test']

        display(trn)

        display(tst)
            
        self.cv = self.final_estimator_.crossvalidate(
                trn, tst,
                dataset_name = str(self.datasets_names_), 
                n_splits = 10,
                n_repeats = 1,
                clear = clear,
                use_tqdm = use_tqdm,
                verbose = verbose,
                name = name,
        )
        return   self.cv

# EXAMPLE
estimator = StackClassifier(
    estimators = [
        ClassifierWrapper(LGBMClassifier(n_estimators=100, random_state=42, verbose=-1, objective='binary', metric='auc'), name = 'LGBM'),
        ClassifierWrapper(XGBClassifier(n_estimators=100, random_state=42, enable_categorical=True, objective='binary:logistic'), name = 'XGB'),
    ],
    datasets_names = 'lxe',
    final_estimator = ClassifierWrapper(LogisticRegression(penalty='l2',  C=10, max_iter=5000, tol=1e-6), name='LR'),
    name = 'STK_LR'
)

# &#128295; Execution settings

In [ ]:
EVALUATE_DATASETS = True
EVALUATE_MODELS   = True
EDA               = True
SUBMIT            = True
OPTUNA            = False
DEBUG             = False

DB                = Database('DATABASE (1)')
LOAD_DB           = True
SHOW_CVs          = True

if LOAD_DB :
    DB.load()
DB

# &#128204; *Models*

In [ ]:
ESTIMATORS = {} # for cv load / rerun

In [ ]:
if AUTOGLUON :
    
    from autogluon.tabular import TabularPredictor
    
    class AGClassifier(Classifier):
    
        def __init__(self, name : str = 'AGC', params : dict = {}, verbose : int = 1, time_limit : int = 300, eval_metric : str = 'roc_auc') :
            super().__init__(name=name, params=params, verbose=verbose)
            
            self.time_limit_ = time_limit
            self.eval_metric_ = eval_metric
            
        def fit(self, X, y, X_val, y_val):
            X[TARGET]  = y
            
            self.model_ = TabularPredictor(
                label = TARGET,
                eval_metric = self.eval_metric_,
                problem_type = 'binary',
            ).fit(
                X,
                presets = 'best_quality',
                time_limit = self.time_limit_,
                verbosity = self.verbose_,
                ag_args_fit = {'num_cpus': 4},
            )
            
        def predict(self, X):
            pred = self.model_.predict(X)
            print('pred:', pred)
            return pred    
            
        def predict_proba(self, X):
            proba = self.model_.predict_proba(X)
            print('proba:', proba)
            return proba.loc[:, 1]    
        
    
    class AGRegressor(Regressor):
    
        def __init__(self, name : str = 'AGR', params : dict = {}, verbose : int = 1, time_limit : int = 300, eval_metric : str = 'root_mean_squared_error') :
            super().__init__(name=name, params=params, verbose=verbose)
            
            self.time_limit_ = time_limit
            self.eval_metric_ = eval_metric
            
        def fit(self, X, y, X_val, y_val):
            X[TARGET]  = y
            
            self.model_ = TabularPredictor(
                label = TARGET,
                eval_metric = self.eval_metric_,
                problem_type = 'regression',
            ).fit(
                X,
                presets = 'best_quality',
                time_limit = self.time_limit_,
                verbosity = self.verbose_,
                ag_args_fit = {'num_cpus': 4},
            )
            
    ESTIMATORS['AGC'] = AGClassifier
    ESTIMATORS['AGR'] = AGRegressor

In [ ]:

class LGBRegressor(Regressor):

    def __init__(self, name='LGBR', params={}, verbose=1) :
        super().__init__(name=name, params=params, verbose=verbose)
        
    def fit(self, X, y, X_val, y_val):
        self.model_ = LGBMRegressor(**self.params_)
        self.model_.fit(
            X, y, 
            eval_set=(X_val, y_val),
            callbacks = [log_evaluation(period=100, show_stdv=False), early_stopping(stopping_rounds=200, verbose=self.verbose_)],            
        )      
    

class LGBClassifier(Classifier):

    def __init__(self, name='LGBC', params={}, verbose=1) :
        super().__init__(name=name, params=params, verbose=verbose)
        
    def fit(self, X, y, X_val, y_val):
        self.model_ = LGBMClassifier(**self.params_)
        self.model_.fit(
            X, y, 
            eval_set=(X_val, y_val),
            callbacks = [log_evaluation(period=100, show_stdv=False), early_stopping(stopping_rounds=200, verbose=self.verbose_)],            
        )      

    
ESTIMATORS['LGBC'] = LGBClassifier
ESTIMATORS['LGBR'] = LGBRegressor    

In [ ]:
class CATRegressor(Regressor):

    def __init__(self, name='CATR', params={}, verbose=1) :
        super().__init__(name=name, params=params, verbose=verbose)

        self.evals_results_ = []
        
    def fit(self, X, y, X_val = None, y_val = None) :
        if X_val is not None :
            cat_features = fcn(X)[1] # X.columns.tolist()            
            train_pool = Pool(data=X, label=y, cat_features=cat_features)
            valid_pool = Pool(data=X_val, label=y_val, cat_features=cat_features)
            self.model_ = catboost.CatBoostRegressor(**self.params_)
            self.model_.fit(train_pool, eval_set=valid_pool, verbose=20)
        else:
            cat_features = X.columns.tolist()            
            train_pool = Pool(data=X, label=y, cat_features=cat_features)
            params =  self.model_.get_params()
            params.update({
                'iterations'     : int(self.model_.tree_count_ * 1.2),
            })
    
            new_params = {}
            for k, v in params.items() :
                if not k in ['use_best_model', 'od_type', 'od_wait', 'early_stopping_rounds'] :
                    new_params[k] = v
                    
            print('\n--- FIT ---', new_params, '---\n')
                    
            self.model_ = catboost.CatBoostRegressor(**new_params)
            self.model_.fit(train_pool, verbose=200)   

        self.evals_results_.append(self.model_.get_evals_result())

    def fit_predict(self, X, y, X_val, y_val):
        self.fit(X, y, X_val, y_val)
        return self.predict(X_val)
        
    def predict(self, X):
        best_iter = self.model_.best_iteration_
        return self.model_.predict(X, ntree_end=best_iter)

    def plot_metrics_(self) :
        if self.evals_results_  == [] :
            print('no metrics to plot')
            return
    
        data, metrics_names = [], set()
        for fold, results in enumerate(self.evals_results_) :
            for curve, metrics in results.items() :
                for metric, values in metrics.items() :
                    metrics_names.add(metric)
                    for iter, value in enumerate(values) :
                        item = {
                            'fold'      : fold,
                            'iteration' : iter,
                            'curve'     : curve,
                             metric     : value,
                        }   
                        data.append(item)
    
        df = pd.DataFrame(data)
        nrows = len(metrics_names)
        fig, ax = plt.subplots(nrows, 1, figsize = (16, 6 * nrows))
        if nrows > 1 :
            ax = ax.flatten()
        else :
            ax = [ax]
        for i, metric in enumerate(metrics_names) :
            sns.lineplot(df, x = 'iteration', y = metric, hue = 'curve', style = 'curve', markers = False, ax = ax[i])
            ax[i].set_ylabel(metric)
            # ax[i].set_xticks(range(df['epoch'].max() + 1))
        plt.tight_layout()
        plt.show()
    
    def display_cv_plots(self) -> None :  
        super().display_cv_plots()
        self.plot_metrics_()
        
ESTIMATORS['CATR'] = CATRegressor

In [ ]:
class CATClassifier(Classifier):

    def __init__(self, name='CATC', params={}, verbose=1) :
        super().__init__(name=name, params=params, verbose=verbose)

        self.evals_results_ = []
        
    def fit(self, X, y, X_val = None, y_val = None) :
        if X_val is not None :
            if 'cat_features' in self.params_ :
                cat_features = self.params_['cat_features']
            else :
                cat_features = X.columns.tolist()            
            train_pool = Pool(data=X, label=y, cat_features=cat_features)
            valid_pool = Pool(data=X_val, label=y_val, cat_features=cat_features)
            self.model_ = catboost.CatBoostClassifier(**self.params_)
            self.model_.fit(train_pool, eval_set=valid_pool, verbose=200)
        else:
            cat_features = X.columns.tolist()            
            train_pool = Pool(data=X, label=y, cat_features=cat_features)
            params =  self.model_.get_params()
            params.update({
                'iterations'     : int(self.model_.tree_count_ * 1.2),
            })
    
            new_params = {}
            for k, v in params.items() :
                if not k in ['use_best_model', 'od_type', 'od_wait', 'early_stopping_rounds'] :
                    new_params[k] = v
                    
            print('\n--- FIT ---', new_params, '---\n')
                    
            self.model_ = catboost.CatBoostClassifier(**new_params)
            self.model_.fit(train_pool, verbose=200)   

        self.evals_results_.append(self.model_.get_evals_result())

    def fit_predict_proba(self, X, y, X_val, y_val):
        self.fit(X, y, X_val, y_val)
        return self.predict_proba(X_val)
        
    def predict_proba(self, X):
        best_iter = self.model_.best_iteration_
        return self.model_.predict_proba(X, ntree_end=best_iter)[:,-1]

    def plot_metrics_(self) :
        if self.evals_results_  == [] :
            print('no metrics to plot')
            return
    
        data, metrics_names = [], set()
        for fold, results in enumerate(self.evals_results_) :
            for curve, metrics in results.items() :
                for metric, values in metrics.items() :
                    metrics_names.add(metric)
                    for iter, value in enumerate(values) :
                        item = {
                            'fold'      : fold,
                            'iteration' : iter,
                            'curve'     : curve,
                             metric     : value,
                        }   
                        data.append(item)
    
        df = pd.DataFrame(data)
        nrows = len(metrics_names)
        fig, ax = plt.subplots(nrows, 1, figsize = (16, 6 * nrows))
        if nrows > 1 :
            ax = ax.flatten()
        else :
            ax = [ax]
        for i, metric in enumerate(metrics_names) :
            sns.lineplot(df, x = 'iteration', y = metric, hue = 'curve', style = 'curve', markers = False, ax = ax[i])
            ax[i].set_ylabel(metric)
            # ax[i].set_xticks(range(df['epoch'].max() + 1))
        plt.tight_layout()
        plt.show()
    
    def display_cv_plots(self) -> None :  
        super().display_cv_plots()
        self.plot_metrics_()

ESTIMATORS['CATC'] = CATClassifier

In [ ]:
import xgboost as xgb

class XGB():
        
    def fit(self, X, y, X_val, y_val):
        dtrain = xgb.DMatrix(X.to_numpy(), label=y.to_numpy())
        dvalid = xgb.DMatrix(X_val.to_numpy(), label=y_val.to_numpy())
        eval_set = [(dtrain, 'train'), (dvalid, 'valid')]        
        self.model_ = xgb.train(
            params=self.params_,
            dtrain=dtrain,
            num_boost_round=5000,
            maximize=True,
            evals=eval_set,
            early_stopping_rounds=300,
            verbose_eval=200,
        )
        
    def predict(self, X):
        return np.clip(self.model_.predict(xgb.DMatrix(X.to_numpy())), a_min = 0,a_max = None)

    def predict_proba(self, X):
        return self.model_.predict(xgb.DMatrix(X.to_numpy()))

In [ ]:
class xGBClassifier(Classifier, XGB):

    def __init__(self, name='XGBC', params={}, verbose=1) :
        Classifier.__init__(self, name=name, params=params, verbose=verbose)
        
    def fit(self, X, y, X_val, y_val):
        XGB.fit(self, X, y, X_val, y_val)
              
    def predict_proba(self, X):
        return XGB.predict_proba(self, X)
    
    
class xGBRegressor(Regressor, XGB):

    def __init__(self, name='XGBR', params={}, verbose=1) :
        Regressor.__init__(self, name=name, params=params, verbose=verbose)
        
    def fit(self, X, y, X_val, y_val):
        XGB.fit(self, X, y, X_val, y_val)
              
    def predict(self, X):
        return XGB.predict(self, X)    
ESTIMATORS['XGBC'] = xGBClassifier
ESTIMATORS['XGBR'] = xGBRegressor    

In [ ]:
def fcn(df, cat_types = ['object', 'category', 'bool', 'string']):
    return df.columns.tolist(), df.select_dtypes(include=cat_types).columns.tolist(), df.select_dtypes(exclude=cat_types).columns.tolist()   

In [ ]:
DATASETS_SCORES = {}

In [ ]:
def display_scores(scores, x='CV', y=SCORE_NAME, plot=True):
    if scores == {}:
        print('scores list is empty')
        return
    print('\nCOMPARE:')
    
    data = []
    for cv_, cv_scores_ in scores.items() :
        item = {'CV' : cv_}
        for k, v in cv_scores_.items():
            item[k] = v
        data.append(item)
        
    df = pd.DataFrame(data)
    display(df.style.background_gradient(subset=df.select_dtypes(include=[float, int]).columns.to_list(), cmap='Greens'))

    if not plot:
        return
    mn, mx = df[y].min(), df[y].max()
    fig, ax = plt.subplots(1, 1, figsize=(15, 5))
    sns.barplot(df, x=y, y=x, ax=ax)
    ax.set(xlim=(mn*0.999, mx*1.001))
    ax.bar_label(ax.containers[0])
    plt.tight_layout()
    plt.show()

In [ ]:
def tocat(src, dst):     
    train, test = DB.datasets.train(src), DB.datasets.test(src)
    cats = fcn(test)[1]
    for df in [train, test] :
        df[cats] = df[cats].astype('string').fillna('--missed--')
    DB.datasets.put(dst, train, test)

def allcat(src, dst):     
    train, test = DB.datasets.train(src), DB.datasets.test(src)
    cats = fcn(test)[0]
    for df in [train, test] :
        df[cats] = df[cats].astype('string').fillna('--missed--')
    DB.datasets.put(dst, train, test)

def enccat(src, dst):     
    train, test = DB.datasets.train(src), DB.datasets.test(src)
    _, CATS, NUMS = fcn(test)
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    
    X_oe = pd.DataFrame(oe.fit_transform(train[CATS]), columns = CATS).fillna(0).astype(int)
    
    test_oe = pd.DataFrame(oe.transform(test[CATS]), columns = CATS).fillna(0).astype(int)
    
    train_e = pd.concat([X_oe, train[NUMS]], axis= 1)
    test_e = pd.concat([test_oe, test[NUMS]], axis= 1)
    
    train_e[TARGET] = train[TARGET]    
    DB.datasets.put(dst, train_e, test_e)


def pipe(src, dst, funcs):
    tmp = src
    for func in funcs :
        print(tmp, func.__name__, dst)
        func(tmp, dst)
        tmp = dst

In [ ]:
def evaluate_dataset(name, clear=True, db=DB, update=False):
    tmp = 'tmp'
    if EVALUATE_DATASETS:
            
        params = {
            'random_state'       : 42,
        }
        estimator = LGBRegressor(params=params)

        cv_name = f'dataset_eval_{name}'

        cv = None
        if db is not None and not update:
            cv = db.cvs.get(cv_name)
            if cv is not None:
                if cv.summary['estimator_class'] != estimator.name or cv.summary['params'] != params :
                    cv = None
                else :
                    estimator.upload_cv(cv)
                    estimator.display_cv_results()
                    estimator.display_cv_plots()
        if cv is None :
            print('encoding categorical features ...', end=' ')
            enccat(name, tmp)
            print(name, 'dataset evaluation ...', end=' ')
            cv = estimator.crossvalidate(
                DB.datasets.train(tmp), 
                DB.datasets.test(tmp), 
                n_splits = 3, 
                dataset_name=name, 
                clear=clear, 
                verbose=2,
                name=cv_name,
            )
            if db is not None:
                print('saving ...', end=' ')
                db.cvs.put(cv)
                # db.save()
                print('ready\n')
        
        DATASETS_SCORES[name] = estimator.cv_scores()
        
        # display_scores(DATASETS_SCORES, plot=False)

In [ ]:
def summary(df, tab=True, plots=False, info=True, nrows=3):
    if tab:
        display(df.head(nrows))
        display(df.tail(nrows))
        print(colored(f'dataframe has {df.shape[0]} rows, {df.shape[1]} columns\n', 'blue'))

    inf = pd.DataFrame(df.dtypes).reset_index().rename(columns={'index':'column', 0:'type'})
    df_missed = pd.DataFrame(df.isnull().sum()).reset_index().rename(columns={'index':'column', 0:'missed'})
    df_unique = pd.DataFrame(df.nunique()).reset_index().rename(columns={'index':'column', 0:'unique'})
    inf['missed'] = df_missed['missed']
    inf['unique'] = df_unique['unique']
    inf['duplicate'] = df.duplicated().sum()
    
    desc = pd.DataFrame(df.describe(include='all').transpose())
    if 'min' in desc.columns.tolist():
        inf['min'] = desc['min'].values
        inf['max'] = desc['max'].values
        inf['avg'] = desc['mean'].values
        inf['std dev'] = desc['std'].values
    if 'top' in desc.columns.tolist():
        inf['top value'] = desc['top'].values
        inf['Freq'] = desc['freq'].values    
    
    if info:
        display(inf.style.background_gradient(subset='missed', cmap='Reds').background_gradient(subset='unique', cmap='Greens'))
  
    if plots:
        print()
        if df_missed['missed'].sum() > 0:
            fig, ax = plt.subplots(1, 1, figsize=(24, 5))
            sns.barplot(df_missed[df_missed['missed'] > 0], x='column', y='missed', ax=ax)
            ax.set_title('missed values') 
            ax.bar_label(ax.containers[0])
            plt.tight_layout()
            plt.show()

        fig, ax = plt.subplots(1, 1, figsize=(24, 5))
        sns.barplot(df_unique[df_unique['unique'] > 0], x='column', y='unique', ax=ax)
        ax.set_title('unique values')
        ax.bar_label(ax.containers[0])
        plt.tight_layout()
        plt.show()

!pip install dython
from dython.nominal import associations
clear_output()

def df_corr(df, name=''):
    associations_df = associations(df, nominal_columns='all', plot=False)
    corr_matrix = associations_df['corr']
    plt.figure(figsize=(20, 8))
    plt.gcf().set_facecolor('#FFFDD0') 
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
    plt.title(f'{name} Correlation Matrix including Categorical Features')
    plt.show()


# &#128269; LOAD DATASETS

In [ ]:
train, test, original, sample_sub = load_datasets()
DB.datasets.put('loaded', train, test)

In [ ]:
DB.datasets.summary('loaded', plots = True)

In [ ]:
if original is not None:
    summary(original, plots=True)

# &#128202; EDA

<div style="display: block; padding: 3px 0; background-color: #ccffcc; color: black; font-size: 12px; font-weight: bold; border-radius: 12px; text-align: center; width: 100%; box-shadow: inset 4px 4px 8px rgba(255, 255, 255, 0.5), inset -4px -4px 8px rgba(0, 0, 0, 0.3); width:40%; margin:auto">
    The target
</div>

In [ ]:
if EDA:
    df, hue  = train, None
    if original is not None:
        t, o = train.copy(), original.copy()
        t['df'] = 'train'
        o['df'] = 'original'
        df, hue = pd.concat([t, o], axis=0), 'df'    
    if PROBLEM == 'regression':
        f = TARGET
        fig, axs = plt.subplots(2, 2, figsize=(24, 10))
        
        ax = axs[0,0]
        sns.kdeplot(train, x=f, label='train', ax=ax, color=red)
        if original is not None:
            sns.kdeplot(original[f], label='original', ax=ax, color=blue)
        ax.set_title(f'{f} distribution')
        ax.legend()
        
        ax = axs[0,1]
        sns.boxplot(df, y=f, x=hue, ax=ax)
        ax.set_title(f'{f} boxplot')
        
        ax = axs[1,0]
        sns.kdeplot(np.log(train[f]), label='train', ax=ax, color=red)
        if original is not None:
            sns.kdeplot(np.log(original[f]), label='original', ax=ax, color=blue)
        ax.set_title(f'log({f}) distribution')
        ax.legend()
        
        ax = axs[1,1]
        if original is not None:
            sns.boxplot(y=np.log(df[f]), x=df['df'], ax=ax)
        else :
            sns.boxplot(y=np.log(df[f]), ax=ax)
        ax.set_title(f'log({f}) boxplot')
        
        plt.tight_layout()
        plt.show()
    else:
        fig, ax = plt.subplots(1, 1, figsize=(24, 5))
        sns.countplot(train, x=TARGET)
        ax.bar_label(ax.containers[0])
        plt.tight_layout()
        plt.show()


<div style="display: block; padding: 3px 0; background-color: #ccffcc; color: black; font-size: 12px; font-weight: bold; border-radius: 12px; text-align: center; width: 100%; box-shadow: inset 4px 4px 8px rgba(255, 255, 255, 0.5), inset -4px -4px 8px rgba(0, 0, 0, 0.3); width:40%; margin:auto">
    The features
</div>

In [ ]:
def top_values(df : pd.DataFrame, column : str, n_max : int = 10, ax = None) :
    vc = pd.DataFrame(df[column].value_counts()).reset_index()[:n_max]
    sns.barplot(vc, y=column, x='count', ax=ax)
    # titles
    ax.set_title(f'{column} Top {n_max}');
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.bar_label(ax.containers[0])

def counts_hist(df : pd.DataFrame, column : str, ax = None) :
    vc = pd.DataFrame(df[column].value_counts()).reset_index()
    sns.histplot(vc, x='count', ax=ax) # bins=int(vc.shape[0]/10), 
    # titles
    ax.set_title(f'{column} counts histogram');
    ax.set_xlabel('')
    ax.set_ylabel('')

def corr(df, title='Correlation', ax=None):
    associations_df = associations(df, nominal_columns='all', plot=False)
    corr_matrix = associations_df['corr']
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, ax=ax)
    ax.set_title(title)

In [ ]:
def top_target(df : pd.DataFrame, column : str, n_max : int = 10, ascending : bool = False, ax = None) :
    vc = pd.DataFrame(df[column].value_counts()) #.reset_index()
    tc = pd.DataFrame(df[column].loc[df[TARGET] == 1].value_counts())#.reset_index()
    tc['ratio'] = tc['count'] * 100 / vc['count']
    tc = tc.sort_values('ratio', ascending=ascending)
    tc['ratio']  = np.round(tc['ratio'], 1)
    if not ascending :
        d = 'top'
    else :
        d = 'bottom'
    tc = tc[:n_max]
    sns.barplot(tc, y=tc.index, x='ratio', ax=ax)
    # titles
    ax.set_title(f'{column}, {d} {n_max} {TARGET} risk');
    ax.set_xlabel('')
    ax.set_ylabel(f'{TARGET} %')
    ax.bar_label(ax.containers[0])

In [ ]:
def top_target_w(df : pd.DataFrame, column : str, n_max : int = 10, ascending : bool = False, ax = None) :
    vc = pd.DataFrame(df[column].value_counts()) #.reset_index()
    tc = pd.DataFrame(df[column].loc[df[TARGET] == 1].value_counts())#.reset_index()
    tc['ratio'] = tc['count'] * 100 * vc['count'] / df.shape[0]
    tc = tc.sort_values('ratio', ascending=ascending)
    tc['ratio']  = np.round(tc['ratio'], 1)
    if not ascending :
        d = 'top'
    else :
        d = 'bottom'
    tc = tc[:n_max]
    sns.barplot(tc, y=tc.index, x='ratio', ax=ax)
    # titles
    ax.set_title(f'{column}, {d} {n_max} {TARGET} risk');
    ax.set_xlabel('')
    ax.set_ylabel(f'{TARGET} % * Sample %')
    ax.bar_label(ax.containers[0])

In [ ]:
def kde_nums(train, test, original):
    _, _, nums = fcn(test)
    columns = nums
    n_cols = 3
    n_rows = math.ceil(len(columns)/n_cols)
    fig, ax = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*5))
    ax = ax.flatten()

    for i, column in enumerate(columns):
        plot_axes = [ax[i]]

        sns.kdeplot(train[column], label='Train', ax=ax[i], color=red)

        sns.kdeplot(test[column], label='Test', ax=ax[i], color=blue)

        if original is not None:
            sns.kdeplot(original[column], label='Original', ax=ax[i], color=green)

        # titles
        ax[i].set_title(f'{column} Distribution');
        ax[i].set_xlabel(None)

    plt.tight_layout()

In [ ]:
if EDA:
    print('Numerical features:')
    kde_nums(train, test, original)

In [ ]:
def count_cats(train_, test, n_cols=3, tight=True):

    train = train_.copy()
    train['nans'] = train['nans'].astype('string')
    
    columns = fcn(test)[1] + ['nans']
    columns.remove('Policy Start Date')
    if len(columns) == 0:
        return
        
    n_cols = 2
    n_rows = math.ceil(len(columns) * 2 / n_cols)
    fig, ax = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*5))
    ax = ax.flatten()

    for i, column in enumerate(columns):
        n = i * n_cols
        
        vc = pd.DataFrame(train[column].value_counts()[:10]).reset_index()
        sns.barplot(vc, x=column, y='count', ax=ax[n])

        # titles
        ax[n].set_title(column);
        ax[n].set_xlabel('')
        ax[n].set_ylabel('')
        ax[n].bar_label(ax[n].containers[0])

        sns.boxplot(train, y = TARGET, x = column, ax = ax[n + 1])

    if tight:
        plt.tight_layout()
    

In [ ]:
if EDA:
    print('Categorical features')
    count_cats(train, test)

<div style="display: block; padding: 3px 0; background-color: #ccffcc; color: black; font-size: 12px; font-weight: bold; border-radius: 12px; text-align: center; width: 100%; box-shadow: inset 4px 4px 8px rgba(255, 255, 255, 0.5), inset -4px -4px 8px rgba(0, 0, 0, 0.3); width:40%; margin:auto">
    Correlation
</div>

In [ ]:
if EDA:
    df_corr(train[:5000], name='TRAIN')     

# &#128476; PREPROCESS

<div style="display: block; padding: 3px 0; background-color: #ccffcc; color: black; font-size: 12px; font-weight: bold; border-radius: 12px; text-align: center; width: 100%; box-shadow: inset 4px 4px 8px rgba(255, 255, 255, 0.5), inset -4px -4px 8px rgba(0, 0, 0, 0.3); width:40%; margin:auto">
    Set the base for dataset evaluation
</div>

In [ ]:
evaluate_dataset('loaded', update=False) 

<div style="display: block; padding: 3px 0; background-color: #ccffcc; color: black; font-size: 12px; font-weight: bold; border-radius: 12px; text-align: center; width: 100%; box-shadow: inset 4px 4px 8px rgba(255, 255, 255, 0.5), inset -4px -4px 8px rgba(0, 0, 0, 0.3); width:40%; margin:auto">
    Add original
</div>

* *and check if the CV score is better*
* *optional, use original or not it depends*

In [ ]:
def add_original(train_, original_):
    f = train_.columns.tolist()
    original_ = original_.dropna(subset = [TARGET])
    train = pd.concat([train_, original_[f]]).reset_index(drop=True) # , axis=0, ignore_index=True
    return train

In [ ]:
# if original is not None:
#     train['original'] = 'no'
#     test['original'] = 'no'
#     original['original'] = 'yes'
#     train_add = add_original(train, original)  
#     DB.datasets.put('+original', train_add, test)
#     DB.datasets.summary('+original')

In [ ]:
# if original is not None:    
#     evaluate_dataset('+original')

## &#128205; Cleaning

In [ ]:
def clean(src, dst):     
    train, test = DB.datasets.train(src), DB.datasets.test(src)

    for df in [train, test] :
        df['Policy Start Date'] = pd.to_datetime(df['Policy Start Date'])
        df['Year'] = df['Policy Start Date'].dt.year.astype(float)    
        df.drop('Policy Start Date', axis=1, inplace=True)    
    DB.datasets.put(dst, train, test)

clean('loaded', 'cleaned')

DB.datasets.summary('cleaned')

In [ ]:
evaluate_dataset('cleaned', update=False)

## &#128205; Feature engineering

In [ ]:
src, dst = 'cleaned', 'fe'
train, test = DB.datasets.train(src), DB.datasets.test(src)

object_columns = fcn(test)[1]

for df in [train, test] :
    for column in object_columns :
        df[column] = df[column].astype('string')
        df['Year'] = df['Year'].astype(float)

unknown = ['Marital Status', 'Occupation', 'Customer Feedback']
for df in [train, test] :
    for column in  unknown:
        df[column] = df[column].fillna('Unknown')

n_cols, n_rows = len(unknown), 1
fig, ax = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*5))
ax = ax.flatten()
for i, column in enumerate(unknown):
    sns.boxplot(train, y = TARGET, x = column, ax = ax[i])
plt.show()

source = train.copy()
source['df'] = 'source'
median = ['Age', 'Vehicle Age', 'Insurance Duration', 'Annual Income', 'Health Score', 'Previous Claims', 'Credit Score']
for df in [train, test] :
    for column in  median:
        mdn = train[column].median()
        df[column] = df[column].fillna(mdn)
        df['Number of Dependents'] = df['Number of Dependents'].fillna(2.0)
source = pd.concat([source, train])
source['df'] = source['df'].fillna('imputed')

DB.datasets.put(dst, train, test)
DB.datasets.summary(dst)

In [ ]:
evaluate_dataset('fe', update=False) 

In [ ]:
display_scores(DATASETS_SCORES)

In [ ]:
    
from sklearn.decomposition import PCA

def plot_variance(pca, width=10, height=4, dpi=100):
    # Create figure
    fig, axs = plt.subplots(1, 2)
    n = pca.n_components_
    grid = np.arange(1, n + 1)
    # Explained variance
    evr = pca.explained_variance_ratio_
    axs[0].bar(grid, evr)
    axs[0].set(
        xlabel="Component", title="% Explained Variance", ylim=(0.0, 0.2)
    )

    # Cumulative Variance
    cv = np.cumsum(evr)
    axs[1].plot(np.r_[0, grid], np.r_[0, cv], "o-")
    axs[1].set(
        xlabel="Component", title="", ylim=(0.0, 1.0)
    )
    
    # Set up figure
    fig.set(figwidth=width, figheight=height,  dpi=100)
    return axs

def make_pca(train, test, display_loadings=True, plots=True):
    features = test.columns.tolist()
    pca = PCA(2)
    X_pca = pca.fit_transform(train.copy()[features])
    X_test_pca = pca.transform(test.copy()[features])
    component_names = [f"PC{i+1}" for i in range(X_pca.shape[1])]
    X_pca = pd.DataFrame(X_pca, columns=component_names)
    X_test_pca = pd.DataFrame(X_test_pca, columns=component_names)

    if display_loadings:
        loadings = pd.DataFrame(
            pca.components_.T,  # transpose the matrix of loadings
            columns=component_names,  # so the columns are the principal components
            index=features,  # and the rows are the original features
        )
        print('\nLoadings:')
        display(loadings.head(20).style.background_gradient(subset=loadings.columns.to_list(), cmap='Greens'))

    if plots:
        plot_variance(pca)
    return X_pca, X_test_pca

In [ ]:
if EDA:
    train_pca, test_pca = make_pca(DB.datasets.train('fe_enccat'), DB.datasets.test('fe_enccat'))

In [ ]:

def mutual_info(train_df, plot=True):
    train = train_df.copy()
    X_train = train.drop([TARGET], axis=1) # [features]
    y_train = train[TARGET]

    if PROBLEM == 'regression':
        mi_scores = mutual_info_regression(X_train, y_train, random_state=42)
    else:
        mi_scores = mutual_info_classif(X_train, y_train, random_state=42)

    mi_scores = pd.Series(mi_scores, name="MI_score", index=X_train.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    df_mi_scores = pd.DataFrame(mi_scores).reset_index().rename(columns={'index':'feature'})
    display(df_mi_scores.style.background_gradient(subset=['MI_score'], cmap='Reds'))
    plt.figure(figsize=(24, 16))
    d = sns.barplot(y=df_mi_scores['feature'], x=df_mi_scores['MI_score'])
    return mi_scores

if EDA:
    mi_scores = mutual_info(DB.datasets.train('fe_enccat').sample(10000))

# &#128204; CV 

## &#128204; *Datasets*

In [ ]:
# tocat('cleaned', 'cleaned_cat')
# allcat('cleaned', 'cleaned_allcat')
# enccat('cleaned', 'cleaned_enccat')
# DB.save()

In [ ]:
# enccat('fe', 'fe_enccat')
# tocat('fe', 'fe_cat')
# DB.save()

## &#128204; *Saved CVs*

In [ ]:
scores = {}
TRAIN, TEST, TRAINP, TESTP = pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
TRAIN[TARGET] = DB.datasets.y('cleaned')

In [ ]:
RERUN = False

for _, cv in DB.cvs.data_.items() :

    summ = cv.summary
    estimator_class = summ['estimator_class']
    name = summ['name']

    if name.startswith('dataset_eval') :
        continue

    display(summ)

    params = summ['params']
    

    if not RERUN :
        print('\n========== model :', estimator_class, '==========\n')
        estimator = ESTIMATORS[estimator_class](params=params)
        estimator.upload_cv(cv)
        if SHOW_CVs :    
            cv.display(estimator)
        cv_scores = estimator.cv_scores()
        cv_scores['Dataset'] = summ['dataset_name']
        scores[name] = cv_scores
        
        #  oof data for stack and blend
        TRAIN[name], TEST[name] = cv.oof['train'], cv.oof['test']
    else:
        estimator = ESTIMATORS[estimator_class](params=params)
        if type(params) is not dict :
            continue

        print(colored(f"\n{estimator_class}:", 'blue'))
        for  k, v in params.items() :
            print('    ', k, ' '*(30-len(k)), ':', v)

        new_cv = cv.run(estimator, DB)
        DB.cvs.put(new_cv)
        
        TRAIN[name]  = new_cv.oof['train']
        TEST[name] = new_cv.oof['test']        
        
        scores[name] = estimator.cv_scores()
        dbnew.save

    # estimator.submit()
    
display_scores(scores)

## &#128204; *New CV*

In [ ]:
SPLITS, REPEATS = 5, 1

In [ ]:

# name, dataset = 'CAT_0', 'cleaned_allcat'

# params = {
#     'iterations': 1000,
#     # 'learning_rate' : 0.05,
#     'random_seed':42,
#     'use_best_model'    : True,
#     'od_type'           : 'Iter',
#     'od_wait'           : 20,
#     'cat_features'  : fcn(DB.datasets.test(dataset))[1],
# }

# estimator = CATRegressor(params=params)
# cv = estimator.crossvalidate(
#         DB.datasets.train(dataset),
#         DB.datasets.test(dataset),
#         dataset_name = dataset, 
#         n_splits = SPLITS,
#         n_repeats = REPEATS,
#         clear = False,
#         verbose = 2,
#         name = name,
# )
# scores[name] = estimator.cv_scores()

# # compare with saved
# display_scores(scores)

# TRAIN[name]  = cv.oof['train']
# TEST[name] = cv.oof['test']

# DB.cvs.put(cv)
# # DB.save()

# # estimator.submit()

In [ ]:

# name, dataset = 'CAT_1', 'fe_cat'

# params = {
#     'loss_function'     : 'RMSE',
#     'iterations'        : 300,
#     # 'learning_rate'     : 0.05,
#     'random_seed'       : 42,
#     'use_best_model'    : True,
#     'od_type'           : 'Iter',
#     'od_wait'           : 20,
#     'cat_features'  : fcn(DB.datasets.test(dataset))[1],
# }

# estimator = CATRegressor(params=params)
# cv = estimator.crossvalidate(
#         DB.datasets.train(dataset),
#         DB.datasets.test(dataset),
#         dataset_name = dataset, 
#         n_splits = SPLITS,
#         n_repeats = REPEATS,
#         clear = False,
#         verbose = 2,
#         name = name,
# )
# scores[name] = estimator.cv_scores()

# # compare with saved
# display_scores(scores)

# TRAIN[name]  = cv.oof['train']
# TEST[name] = cv.oof['test']

# DB.cvs.put(cv)
# # DB.save()

# # estimator.submit()

In [ ]:

def addbest(src, dst):     
    train, test = DB.datasets.train(src), DB.datasets.test(src)
    train_b, test_b = DB.datasets.train('best'), DB.datasets.test('best')
    train['best'] = train_b['best']
    test['best'] = test_b['best']
    DB.datasets.put(dst, train, test)

In [ ]:

# addbest('fe_cat', 'fe_cat_best')
# addbest('fe_enccat', 'fe_enccat_best')

In [ ]:

# name, dataset = 'CAT_2', 'fe_cat_best'

# params = {
#     'loss_function'     : 'RMSE',
#     'iterations'        : 300,
#     # 'learning_rate'     : 0.05,
#     'random_seed'       : 42,
#     'use_best_model'    : True,
#     'od_type'           : 'Iter',
#     'od_wait'           : 20,
#     'cat_features'  : fcn(DB.datasets.test(dataset))[1],
# }

# estimator = CATRegressor(params=params)
# cv = estimator.crossvalidate(
#         DB.datasets.train(dataset),
#         DB.datasets.test(dataset),
#         dataset_name = dataset, 
#         n_splits = SPLITS,
#         n_repeats = REPEATS,
#         clear = False,
#         verbose = 2,
#         name = name,
# )
# scores[name] = estimator.cv_scores()

# # compare with saved
# display_scores(scores)

# TRAIN[name]  = cv.oof['train']
# TEST[name] = cv.oof['test']

# DB.cvs.put(cv)
# # DB.save()

# # estimator.submit()

In [ ]:

# name, dataset = 'CAT_3', 'fe_cat_best'

# params = {
#     'loss_function'     : 'RMSE',
#     'iterations'        : 3000,
#     # 'learning_rate'     : 0.05,
#     'random_seed'       : 42,
#     'use_best_model'    : True,
#     'od_type'           : 'Iter',
#     'od_wait'           : 20,
#     'cat_features'  : fcn(DB.datasets.test(dataset))[1],
# }

# estimator = CATRegressor(params=params)
# cv = estimator.crossvalidate(
#         DB.datasets.train(dataset),
#         DB.datasets.test(dataset),
#         dataset_name = dataset, 
#         n_splits = SPLITS,
#         n_repeats = REPEATS,
#         clear = False,
#         verbose = 2,
#         name = name,
# )
# scores[name] = estimator.cv_scores()

# # compare with saved
# display_scores(scores)

# TRAIN[name]  = cv.oof['train']
# TEST[name] = cv.oof['test']

# DB.cvs.put(cv)
# # DB.save()

# estimator.submit()

In [ ]:

# name, dataset = 'LGB_0', 'cleaned_enccat'

# params = {
#     # 'n_estimators': 1000,
#     'learning_rate' : 0.05,
#     'random_seed':42,
# }

# estimator = LGBRegressor(params=params)
# cv = estimator.crossvalidate(
#         DB.datasets.train(dataset),
#         DB.datasets.test(dataset),
#         dataset_name = dataset, 
#         n_splits = SPLITS,
#         n_repeats = REPEATS,
#         clear = True,
#         verbose = 2,
#         name = name,
# )
# scores[name] = estimator.cv_scores()

# # compare with saved
# display_scores(scores)

# TRAIN[name]  = cv.oof['train']
# TEST[name] = cv.oof['test']

# DB.cvs.put(cv)
# # DB.save()

# # estimator.submit()

In [ ]:

# name, dataset = 'LGB_1', 'fe_enccat'

# params = {
#     'n_esimators': 3000,
#     'learning_rate' : 0.05,
#     'random_seed':42,
# }

# estimator = LGBRegressor(params=params)
# cv = estimator.crossvalidate(
#         DB.datasets.train(dataset),
#         DB.datasets.test(dataset),
#         dataset_name = dataset, 
#         n_splits = SPLITS,
#         n_repeats = REPEATS,
#         clear = True,
#         verbose = 2,
#         name = name,
# )
# scores[name] = estimator.cv_scores()

# # compare with saved
# display_scores(scores)

# TRAIN[name]  = cv.oof['train']
# TEST[name] = cv.oof['test']

# DB.cvs.put(cv)
# # DB.save()

# # estimator.submit()

In [ ]:

# name, dataset = 'LGB_2', 'fe_enccat_best'

# params = {
#     'n_estimators': 3000,
#     'learning_rate' : 0.05,
#     'random_seed':42,
# }

# estimator = LGBRegressor(params=params)
# cv = estimator.crossvalidate(
#         DB.datasets.train(dataset),
#         DB.datasets.test(dataset),
#         dataset_name = dataset, 
#         n_splits = SPLITS,
#         n_repeats = REPEATS,
#         clear = True,
#         verbose = 2,
#         name = name,
# )
# scores[name] = estimator.cv_scores()

# # compare with saved
# display_scores(scores)

# TRAIN[name]  = cv.oof['train']
# TEST[name] = cv.oof['test']

# DB.cvs.put(cv)
# # DB.save()

# # estimator.submit()

In [ ]:

# name, dataset = 'XGB_0', 'cleaned_enccat'

# params = {
#     # 'n_esimators': 1000,
#     'learning_rate' : 0.05,
#     'random_seed':42,
# }

# estimator = xGBRegressor(params=params)
# cv = estimator.crossvalidate(
#         DB.datasets.train(dataset),
#         DB.datasets.test(dataset),
#         dataset_name = dataset, 
#         n_splits = SPLITS,
#         n_repeats = REPEATS,
#         clear = True,
#         verbose = 2,
#         name = name,
# )
# scores[name] = estimator.cv_scores()

# # compare with saved
# display_scores(scores)

# TRAIN[name]  = cv.oof['train']
# TEST[name] = cv.oof['test']

# DB.cvs.put(cv)
# # DB.save()

# # estimator.submit()

In [ ]:

# name, dataset = 'XGB_1', 'fe_enccat'

# params = {
#     'n_estimators': 5000,
#     'learning_rate' : 0.05,
#     'random_seed':42,
# }

# estimator = xGBRegressor(params=params)
# cv = estimator.crossvalidate(
#         DB.datasets.train(dataset),
#         DB.datasets.test(dataset),
#         dataset_name = dataset, 
#         n_splits = SPLITS,
#         n_repeats = REPEATS,
#         clear = True,
#         verbose = 2,
#         name = name,
# )
# scores[name] = estimator.cv_scores()

# # compare with saved
# display_scores(scores)

# TRAIN[name]  = cv.oof['train']
# TEST[name] = cv.oof['test']

# DB.cvs.put(cv)
# # DB.save()

# # estimator.submit()

In [ ]:

# name, dataset = 'XGB_2', 'fe_enccat_best'

# params = {
#     'n_estimators': 5000,
#     'learning_rate' : 0.05,
#     'random_seed':42,
# }

# estimator = xGBRegressor(params=params)
# cv = estimator.crossvalidate(
#         DB.datasets.train(dataset),
#         DB.datasets.test(dataset),
#         dataset_name = dataset, 
#         n_splits = SPLITS,
#         n_repeats = REPEATS,
#         clear = True,
#         verbose = 2,
#         name = name,
# )
# scores[name] = estimator.cv_scores()

# # compare with saved
# display_scores(scores)

# TRAIN[name]  = cv.oof['train']
# TEST[name] = cv.oof['test']

# DB.cvs.put(cv)
# DB.save()

# # estimator.submit()

## &#128204; *OOF Data*

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(TRAIN.corr(), annot=True, fmt='.2f')
plt.show()

In [ ]:
df = pd.DataFrame()
for c in TEST.columns.tolist() :
    x = pd.DataFrame()
    x['oof_proba'] = TRAIN[c]
    x['model'] = c
    df = pd.concat([df, x], axis=0)
                   
plt.figure(figsize=(16, 5))
sns.kdeplot(TEST, bw_adjust=0.1, log_scale=(False, False))
plt.show()

In [ ]:
df = TRAIN.copy()
X_train = df.drop([TARGET], axis=1) # [features]
y_train = df[TARGET]

if PROBLEM == 'regression':
    mi_scores = mutual_info_regression(X_train, y_train, random_state=42)
else:
    mi_scores = mutual_info_classif(X_train, y_train, random_state=42)

mi_scores = pd.Series(mi_scores, name="MI_score", index=X_train.columns)
mi_scores = mi_scores.sort_values(ascending=False)
df_mi_scores = pd.DataFrame(mi_scores).reset_index().rename(columns={'index':'feature'})
display(df_mi_scores.style.background_gradient(subset=['MI_score'], cmap='Reds'))
plt.figure(figsize=(24, 16))
d = sns.barplot(y=df_mi_scores['feature'], x=df_mi_scores['MI_score'])


## &#128204; STACK
#### members list otimization

In [ ]:
import itertools
def best_members(members, f_, constant_members=[], n_min=2, n_max=0, verbose=1, name='estimator'):
    
    def f(x) :
        return OBJ * f_(x)
    
    best, f_best = [], -np.inf
    
    all, history = [], []
    stop = len(members) + 1
    if n_max > 0:
        stop = n_max
    for i in range(n_min, stop):
        all_i = list(itertools.combinations(members, i))
        all.extend(all_i)
    trials = len(all)
        
    trial = 1
    for curr in all:
        is_best = False
        curr = list(curr)
        curr.extend(constant_members)
        f_curr = f(curr)
        if verbose > 0:
            print(f'{trial}/{trials}', curr, ' '*(50-len(str(curr))), '->', OBJ * f_curr, end=' ')
        if f_curr > f_best:
            f_best = f_curr
            best = curr
            is_best = True
            if verbose > 0:
                print('best so far', end=' ')
        if verbose > 0:
            print()
        history.append({
            'trial'    : trial,
            'members'  : str(curr),
            'best'     : is_best,
            SCORE_NAME : OBJ * f_curr,
            'estimator': name,
        })
        trial += 1
    if verbose > 0:
        print('\n\nthe best:\n', best, '->', OBJ * f_best)
    return best, f_best, history

In [ ]:
def stack(train, test, members, verbose=2, name='STACK', submission=False):
    
    # estimator = RegressorWrapper(LinearRegression(), name=name)
    # estimator = xGBRegressor(params={
    estimator = LGBRegressor(params={
        'n_estimators': 10000,
        # 'learning_rate' : 0.01,
        'random_seed':42,    
    })
    cv = estimator.crossvalidate(
        train[members + [TARGET]], 
        test[members], 
        dataset_name=str(members), 
        n_splits=5,
        n_repeats=1,
        verbose=verbose,
        clear=False,
        name=name,
    )
    if submission:
        estimator.submit()
    return estimator.cv_scores(), cv.oof['train'], cv.oof['test']

def stack_score(members):
    scores, _, _ = stack(TRAIN, TEST, members, verbose=0)
    return scores[SCORE_NAME]

In [ ]:
members = TEST.columns.tolist() 
members

In [ ]:
# best, best_score, stack_history  = best_members(members, stack_score, name = 'linear regression')

In [ ]:
score, train_best, test_best = stack(TRAIN, TEST, members, submission=True)
scores['STACK'] = score

In [ ]:
# trn, tst = pd.DataFrame(), pd.DataFrame()
# trn['best'] = train_best
# tst['best'] = test_best
# DB.datasets.put('best', trn, tst)
# DB.save()
# DB

## &#128204; BLEND *differential evolution*
#### members list and weights otimization

In [ ]:
def blend_diff(train, test, members, verbose=1):
    
    def weighted_average(weights, values):
        qty = len(values)
        sum_values = values[0] * weights[0]
        sum_weights = weights[0]
        for i in range(1, qty):
            sum_values += values[i] * weights[i]
            sum_weights += weights[i]
        return sum_values / sum_weights

    def obj(weights):
        preds = weighted_average(weights, X)
        return LOSS(y, preds)    
 
    X = [train[col].values for col in members]
    if verbose > 0:
        print('train arrays:')
        for a in X:
            print(a)
            
    maxiter = len(members) * 1000
    
    y = train[TARGET].values
    
    qty = len(X)
    initial_weights = [1 for _ in range(qty)]
    bounds = [(0.0, 1.0) for _ in range(qty)]
    result = sp.optimize.differential_evolution(obj, bounds=bounds, maxiter=1000*qty)
    weights = result.x
    
    if verbose > 0 :
        print('\n', obj(initial_weights), end=' => ')
        print(weights)
        print() 
        
    score = SCORE(y, weighted_average(weights, X))
    if verbose > 0:
        print(colored(f'estimated {SCORE_NAME} : {score}\n\n', 'red'))

    X_test = [test[col].values for col in members]
    if verbose > 0:
        print('arrays to blend:')
        for a in X_test:
            print(a)

    result = weighted_average(weights, X_test)
    if verbose > 0:
        print('weighted average:\n', result, '\n')
    
    return result, score

def blend_score_diff(members):
    _, score = blend_diff(TRAIN, TEST, members, verbose=0)
    return score

In [ ]:
members[-3:]

In [ ]:
# best, best_score, diff_history  = best_members(members, blend_score_diff, name='differential evolution')

In [ ]:
TEST_PREDS_D, score_diff = blend_diff(TRAIN, TEST, members) 

In [ ]:
scores['DIFF_EV'] = {
    'Model' : 'DIFF_EV',
    SCORE_NAME : score_diff,
}

# TEST['DIFF'] = TEST_PROBAS_D

In [ ]:
sub = sample_sub.copy()
sub[TARGET] = TEST_PREDS_D
sub.to_csv(f"diff_ev_{score_diff}.csv", index=False)
display(sub.head(10))

## &#128204; BLEND *sp.optimize.minimize*
#### members list and weights otimization

In [ ]:
def blend(train, test, members, verbose=1):
    X = [train[col].values for col in members]
    if verbose > 0:
        print('train arrays:')
        for a in X:
            print(a)
            
    maxiter = len(members) * 500
    
    y = train[TARGET].values
    
    averager = Averager(options={'maxiter':maxiter})
    
    x_valid = np.clip(averager.fit_predict(X, y), 20.0, 4999.0)
    
    if verbose > 0:
        print('\nWEIGHTS:\n', averager.weights())
        print('weighted average:\n', x_valid)

    score = SCORE(y, x_valid)
    if verbose > 0:
        print(colored(f'estimated {SCORE_NAME} : {score}\n\n', 'red'))

    X_test = [test[col].values for col in members]
    if verbose > 0:
        print('arrays to blend:')
        for a in X_test:
            print(a)

    result = np.clip(averager.predict(X_test), 20, 4999)
    if verbose > 0:
        print('weighted average:\n', result, '\n')
    
    return result, score

def blend_score(members):
    _, score = blend(TRAIN, TEST, members, verbose=0)
    return score

In [ ]:
# best, best_score, blend_history = best_members(members, blend_score, name='sp.optimize.minimize')

In [ ]:
TEST_PREDS_A, score_averager = blend(TRAIN, TEST, members) 

In [ ]:
scores['BLEND'] = {
    'Model' : 'BLEND',
    SCORE_NAME : score_averager,
}
display_scores(scores)

## &#128204; Compare ensembles

In [ ]:

# dfs = pd.DataFrame(stack_history)
# dfb = pd.DataFrame(blend_history)
# dfd = pd.DataFrame(diff_history)
# df = pd.concat([dfs, dfb, dfd], axis=0)

# mn, mx = df[SCORE_NAME].min(), df[SCORE_NAME].max()
# fig, axs = plt.subplots(2, 1, figsize=(16, 18))
# axs = axs.flatten()

# ax = axs[0]
# sns.barplot(df, y='members', x=SCORE_NAME, hue='estimator', palette=[green, red, blue, black], ax=ax)
# ax.set(xlim=(mn*0.9999, mx*1.0001))
# ax.legend = False

# ax = axs[1]
# sns.lineplot(df, x='trial', y=SCORE_NAME, hue='estimator', palette=[green, red, blue, black], ax=ax)
# ax.set(ylim=(mn*0.9999, mx*1.0001))
# plt.tight_layout()
# plt.show()

### &#128190; SUBMISSION

In [ ]:
sub = sample_sub.copy()
sub[TARGET] = TEST_PREDS_A
sub.to_csv(f"averager_{score_averager}.csv", index=False)
display(sub.head(10))

In [ ]:
DB.save()
DB